# Transformation of Netflix data

Logging Pipeline

In [2]:
from pyspark.sql.functions import current_timestamp

def log_to_table(step, status, row_count, message):
    log_data = [( 
        "netflix-pipeline",  
        step,
        status,
        row_count,
        message
    )]

    log_df = spark.createDataFrame(
        log_data,
        ["pipeline_name", "step", "status", "row_count", "message"]
    ).withColumn("timestamp", current_timestamp())

    log_df.write.format("delta") \
        .mode("append") \
        .save("Tables/pipeline_logs")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 4, Finished, Available, Finished, False)

In [3]:
log_to_table("START", "SUCCESS", 0, "Tranformation of netflix data started...")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 5, Finished, Available, Finished, False)

Read data as a spark dataframe

In [4]:
df = spark.read.option("header", True) \
    .csv("Files/netflix/bronze/netflix_data.csv")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 6, Finished, Available, Finished, False)

In [5]:
row_count = df.count()

log_to_table("READ_BRONZE", "SUCCESS", row_count, "Data loaded from bronze")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 7, Finished, Available, Finished, False)

In [6]:
df.show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 8, Finished, Available, Finished, False)

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

Handling Duplicate Values

In [7]:
df = df.dropDuplicates()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 9, Finished, Available, Finished, False)

Handling Null Values

In [8]:
df = df.fillna({
    "country": "Unknown",
    "rating": "Not Rated",
    "director": "Unknown"
})

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 10, Finished, Available, Finished, False)

Handling Trailing Spaces

In [9]:
from pyspark.sql.functions import *

df = df.withColumn("country", trim(col("country")))

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 11, Finished, Available, Finished, False)

In [10]:
clean_count = df.count()

log_to_table("TRANSFORM_SILVER", "SUCCESS", clean_count, "Data cleaned")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 12, Finished, Available, Finished, False)

Save transformed data in Silver Layer in Lakehouse

In [11]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_netflix")

df.write.format("delta") \
    .mode("overwrite") \
    .save("Files/netflix/silver/silver_netflix")
    
log_to_table("WRITE_SILVER", "SUCCESS", clean_count, "Transformed data stored in the Silver Layer.")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 13, Finished, Available, Finished, False)

### Create Gold Layer

1. Movies vs TV Shows

In [12]:
df_content_type = df.groupBy("type").count()

df_content_type.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_content_type")


StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 14, Finished, Available, Finished, False)

In [13]:
df_content_type.show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 15, Finished, Available, Finished, False)

+-------------+-----+
|         type|count|
+-------------+-----+
|      TV Show| 2676|
|        Movie| 6131|
|         NULL|    1|
|William Wyler|    1|
+-------------+-----+



2. Content by Country

In [14]:
df_country = df.groupBy("country").count()

df_country.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_country")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 16, Finished, Available, Finished, False)

In [15]:
df_country.show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 17, Finished, Available, Finished, False)

+--------------------+-----+
|             country|count|
+--------------------+-----+
|India, United Kin...|    1|
|Chile, United Sta...|    1|
|              Russia|   15|
|Denmark, France, ...|    1|
|United States, Po...|    1|
|United States, Ir...|    3|
|United Kingdom, N...|    2|
|Brazil, France, G...|    1|
|Australia, United...|    2|
| India, Soviet Union|    2|
|Brazil, United St...|    1|
|France, United St...|   10|
|Turkey, United St...|    2|
|United Kingdom, C...|    1|
|              Sweden|   13|
|Italy, United Kin...|    1|
|South Korea, Czec...|    1|
|       Turkey, India|    1|
|Spain, United Kin...|    2|
|        China, Japan|    1|
+--------------------+-----+
only showing top 20 rows



3. Year-wise Trend

In [16]:
from pyspark.sql.functions import year, to_date, col

df_year = df.withColumn("date_added", to_date(col("date_added"), "MMMM d, yyyy")) \
            .withColumn("release_year", year(col("date_added"))) \
            .groupBy("release_year") \
            .count() \
            .orderBy("release_year")

df_year.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_yearly_trend")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 18, Finished, Available, Finished, False)

In [17]:
df_year.show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 19, Finished, Available, Finished, False)

+------------+-----+
|release_year|count|
+------------+-----+
|        NULL|  120|
|        2008|    2|
|        2009|    2|
|        2010|    1|
|        2011|   13|
|        2012|    3|
|        2013|   10|
|        2014|   23|
|        2015|   72|
|        2016|  418|
|        2017| 1162|
|        2018| 1623|
|        2019| 1997|
|        2020| 1872|
|        2021| 1491|
+------------+-----+



4. Top Ratings

In [18]:
df_rating = df.groupBy("rating").count()

df_rating.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_rating")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 20, Finished, Available, Finished, False)

In [19]:
df_rating.show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 21, Finished, Available, Finished, False)

+----------------+-----+
|          rating|count|
+----------------+-----+
|       Not Rated|    6|
|   Adriane Lenox|    1|
|            TV-Y|  307|
|            2019|    1|
|              UR|    3|
|              PG|  286|
|     Jide Kosoko|    1|
|           TV-MA| 3195|
| Jowharah Jones"|    1|
|              NR|   80|
|           TV-PG|  862|
|               R|  796|
|               G|   41|
|            2021|    2|
|           TV-14| 2158|
|            TV-G|  220|
| Kristen Schaal"|    1|
|           TV-Y7|  334|
|           PG-13|  489|
|   Maury Chaykin|    1|
+----------------+-----+
only showing top 20 rows



In [20]:
log_to_table("WRITE_GOLD", "SUCCESS", 4, "Gold tables created.")

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 22, Finished, Available, Finished, False)

In [21]:
spark.sql("SHOW TABLES").show()

StatementMeta(, faf37650-77cd-47b4-a72a-1135dcf770ac, 23, Finished, Available, Finished, False)

+--------------------+-----------------+-----------+
|           namespace|        tableName|isTemporary|
+--------------------+-----------------+-----------+
|`rc-project`.netf...|gold_content_type|      false|
|`rc-project`.netf...|     gold_country|      false|
|`rc-project`.netf...|      gold_rating|      false|
|`rc-project`.netf...|gold_yearly_trend|      false|
|`rc-project`.netf...|   silver_netflix|      false|
+--------------------+-----------------+-----------+

